# DAG Data Pipeline | DAG-based Parallel Execution

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

class DAGState(TypedDict):
    raw_data: str
    config: NotRequired[str]
    extracted: NotRequired[str]
    validated: NotRequired[str]
    transformed: NotRequired[str]
    report: NotRequired[str]

In [4]:
# Task A: Extract key data points (depends on: input)
def extract_data(state: DAGState) -> dict:
    response = model.invoke(
        f"Extract all numerical data points, dates, and named entities from this text:\n\n{state['raw_data']}"
    )
    return {"extracted": response.content}

# Task B: Determine processing config (depends on: input) -- runs in parallel with A
def fetch_config(state: DAGState) -> dict:
    response = model.invoke(
        f"Analyze this text and determine the best processing approach:\n"
        f"- What type of document is this? (financial, technical, legal, general)\n"
        f"- What analysis would be most valuable?\n"
        f"- What format should the output take?\n\n{state['raw_data']}"
    )
    return {"config": response.content}

In [5]:
# Task C: Transform data (depends on: A and B)
def transform_data(state: DAGState) -> dict:
    response = model.invoke(
        f"Transform and enrich the extracted data based on the processing config.\n\n"
        f"Extracted Data:\n{state['extracted']}\n\n"
        f"Processing Config:\n{state['config']}"
    )
    return {"transformed": response.content}

# Task D: Validate schema (depends on: A) -- runs in parallel with C
def validate_schema(state: DAGState) -> dict:
    response = model.invoke(
        f"Validate the extracted data for completeness and consistency. "
        f"Flag any missing fields, contradictions, or anomalies:\n\n{state['extracted']}"
    )
    return {"validated": response.content}

In [6]:
# Task E: Generate report (depends on: C and D)
def generate_report(state: DAGState) -> dict:
    response = model.invoke(
        f"Generate a final analysis report.\n\n"
        f"Transformed Data:\n{state['transformed']}\n\n"
        f"Validation Results:\n{state['validated']}\n\n"
        f"Create a structured report with findings, quality notes, and recommendations."
    )
    return {"report": response.content}

In [7]:
# Build the DAG
graph = StateGraph(DAGState)
graph.add_node("extract", extract_data)
graph.add_node("config", fetch_config)
graph.add_node("transform", transform_data)
graph.add_node("validate", validate_schema)
graph.add_node("report", generate_report)

# Define edges to create the DAG structure
graph.add_edge(START, "extract")          # A starts immediately
graph.add_edge(START, "config")           # B starts immediately (parallel with A)
graph.add_edge("extract", "transform")    # C depends on A
graph.add_edge("config", "transform")     # C depends on B (waits for both)
graph.add_edge("extract", "validate")     # D depends on A (parallel with C)
graph.add_edge("transform", "report")     # E depends on C
graph.add_edge("validate", "report")      # E depends on D (waits for both)
graph.add_edge("report", END)

dag_pipeline = graph.compile()

In [8]:
plot_mermaid(dag_pipeline)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract(extract)
	config(config)
	transform(transform)
	validate(validate)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ --> config;
	__start__ --> extract;
	config --> transform;
	extract --> transform;
	extract --> validate;
	transform --> report;
	validate --> report;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
result = dag_pipeline.invoke({
    "raw_data": "Q3 2024 Revenue: $4.2B (up 12% YoY). Operating margin improved to 28% from 25%. "
                "Key growth drivers: Cloud services (+34%), AI products (+89%). "
                "Headcount reduced by 5% to 12,400 employees. Cash reserves: $18.7B. "
                "Guidance for Q4: Revenue $4.5-4.7B. New partnerships with 3 Fortune 500 companies."
})
print(result["report"])

# Final Analysis Report

## Financial Analysis Report

### Company Overview
This financial analysis report scrutinizes the company's performance during Q3 2024 and anticipates developments for Q4 2024. The analysis centers on revenue growth, profitability, operational efficiency, cash flow, and future outlook, supported by financial data and projections.

---

## 1. Revenue and Growth Analysis

### Key Revenue Data:
- **Q3 2024 Revenue:** $4.2 billion  
- **Year-over-Year Growth:** 12% increase from the previous year  
- **Q4 Revenue Guidance:** $4.5 - $4.7 billion  

### Growth Drivers:
- **Cloud Services Growth:** 34%
- **AI Products Growth:** 89%

#### Insights:
The data reflects a vigorous revenue trajectory, predominantly fueled by Cloud services and AI products. AI product sales stand out with substantial growth. Q4 projections suggest an ongoing growth momentum, with an anticipated revenue increase between 7.1% and 11.9% compared to Q3.

---

## 2. Profitability Analysis

### Ke

In [10]:
# Streaming

stream_invoke(
    dag_pipeline, {
        "raw_data": "Q3 2024 Revenue: $4.2B (up 12% YoY). Operating margin improved to 28% from 25%. "
                    "Key growth drivers: Cloud services (+34%), AI products (+89%). "
                    "Headcount reduced by 5% to 12,400 employees. Cash reserves: $18.7B. "
                    "Guidance for Q4: Revenue $4.5-4.7B. New partnerships with 3 Fortune 500 companies."
    }
)


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'raw_data': 'Q3 2024 Revenue: $4.2B (up 12% YoY). Operating margin improved to 28% from 25%. Key growth drivers: Cloud services (+34%), AI products (+89%). Headcount reduced by 5% to 12,400 employees. Cash reserves: $18.7B. Guidance for Q4: Revenue $4.5-4.7B. New partnerships with 3 Fortune 500 companies.',
 'config': "- **What type of document is this?**  \n  This is a financial document. It contains information related to a company's revenue, operating margin, growth drivers, headcount, cash reserves, future revenue guidance, and new partnerships.\n  \n- **What analysis would be most valuable?**  \n  Valuable analysis would include:\n  1. **Revenue and Growth Analysis:** Evaluating the revenue increase and the specific contributions of cloud services and AI products.\n  2. **Profitability Analysis:** Examining the improvement in operating margin.\n  3. **Efficiency Analysis:** Assessing the implications of headcount reduction on productivity and costs.\n  4. **Cash Flow and Liquidit